# Talyxion SDK — Quickstart

End-to-end notebook covering all 3 surfaces:

1. **Alpha research** — build, simulate, plot PnL, check overfit
2. **Trading desk** — credentials → profiles → cycles → live positions
3. **Marketplace** — browse, buy with VietQR top-up, list-for-sale

**Prerequisites:**
```bash
pip install talyxion pandas matplotlib
export TALYXION_API_KEY=tk_...
```


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from talyxion import Talyxion, Alpha, SuperAlpha

tlx = Talyxion()  # reads TALYXION_API_KEY
print('Connected:', tlx.status())

## 1. Alpha research

Fluent builder lets you chain config without verbose dicts. Terminal `.simulate(client)` hits the server, runs the sim, and saves the alpha to your library if `save=True`.

In [ ]:
result = (
    Alpha('rank(close - ts_mean(close, 20)) * volume')
    .region('crypto_trade')
    .universe('TOP19')
    .delay(1)
    .decay(4)
    .truncation(0.08)
    .simulate(tlx)
)

print(f'alpha_id  = {result.alpha_id}')
print(f'Sharpe    = {result.sharpe:.2f}')
print(f'Fitness   = {result.fitness:.2f}')
print(f'Turnover  = {result.turnover:.3f}')
print(f'Drawdown  = {result.drawdown:.3f}')
print(f'Passes overfit (rule of thumb): {result.passes_overfit()}')

### Plot equity curve + drawdown

`PnLSeries.to_dataframe()` returns a pandas DataFrame with `equity` and `drawdown` columns ready for plotting.

In [ ]:
df = tlx.alphas.pnl(result.alpha_id).to_dataframe()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.index, df['equity'], color='#22d3ee', linewidth=1.6)
ax1.set_ylabel('Equity')
ax1.grid(True, alpha=0.2)
ax2.fill_between(df.index, df['drawdown'], 0, color='#fb923c', alpha=0.4)
ax2.plot(df.index, df['drawdown'], color='#fb923c', linewidth=1)
ax2.set_ylabel('Drawdown')
ax2.grid(True, alpha=0.2)
plt.suptitle(f'{result.alpha_id} — Sharpe {result.sharpe:.2f}')
plt.tight_layout()
plt.show()

### Authoritative overfit checks

5 rules: Sharpe ≥ 1.58, Fitness ≥ 1.0, Turnover 0.01–0.7, Ladder Sharpe (autocorr-adjusted p ≤ 0.05), |Drawdown| ≤ 0.2.

In [ ]:
report = tlx.alphas.overfit(result.alpha_id)
for c in report.checks:
    mark = '✓' if c.passed else '✗'
    print(f'{mark} {c.label:30s} {c.result}')
print(f'\npasses_all = {report.passes_all}')

### Browse my alpha library

In [ ]:
page = tlx.alphas.list(mine_only=True, sort='sharpe', order='desc', limit=20)
df_alphas = page.to_dataframe()
df_alphas[['id', 'sharpe', 'fitness', 'turnover', 'drawdown', 'region']].head(10)

### Super alpha (linear combo)

In [ ]:
top_two = page.items[:2]
if len(top_two) >= 2:
    super_result = (
        SuperAlpha([a.id for a in top_two], combo='0.6 * a + 0.4 * b')
        .region('crypto_trade').universe('TOP19')
        .simulate(tlx)
    )
    print(f'super alpha_id = {super_result.alpha_id}')
    print(f'Sharpe = {super_result.sharpe:.2f}')

## 2. Trading desk

Manage exchange API credentials and trading profiles. Each profile binds an alpha to a credential + risk caps. Cycle dispatcher runs every 60s on the server.

In [ ]:
creds = tlx.trading.credentials.list()
for c in creds:
    print(f'#{c.id}  {c.exchange:12s}  {c.label:10s}  status={c.validation_status}')

In [ ]:
# Add a credential (commented; uncomment with your real keys)
# cred = tlx.trading.credentials.create(
#     exchange='binance', label='main',
#     api_key='YOUR_BINANCE_API_KEY',
#     api_secret='YOUR_BINANCE_API_SECRET',
# )
# tlx.trading.credentials.validate(cred.id)

In [ ]:
profiles = tlx.trading.profiles.list()
for p in profiles:
    print(f'#{p.id}  {p.name:20s}  {p.exchange:10s}  {p.mode:10s}  {p.status}')

### Create + activate a simulation profile

Free tier: 0 profiles · Pro: 1 · Pro+: 3 · Max: 5. Live mode requires Pro+.

In [ ]:
# Uncomment after you have a credential + licensed alpha:
# profile = tlx.trading.profiles.create(
#     name='btc_meanrev_v1',
#     alpha_id=result.alpha_id,
#     exchange='binance',
#     credential_id=creds[0].id,
#     mode='simulation',
#     leverage=2,
#     book_usd=500,
#     cycle_interval_sec=300,
#     max_drawdown_pct=15,
# ).activate()
# print(profile)

### Recent cycles + live positions

In [ ]:
if profiles:
    profile = profiles[0]
    cycles = profile.cycles.tail(20)
    df_cycles = pd.DataFrame([c.model_dump() for c in cycles])
    df_cycles[['started_at', 'outcome', 'trades_attempted', 'trades_filled', 'duration_ms']].head(10)

In [ ]:
if profiles and profiles[0].status == 'active':
    snap = profiles[0].positions()
    print(f'Wallet balance: ${snap.wallet_balance:,.2f}')
    print(f'Unrealized PnL: ${snap.unrealized_pnl:,.2f}')
    print(f'Open positions: {snap.position_count}')
    pd.DataFrame([p.model_dump() for p in snap.positions])

### Lifecycle controls

Profile handles are chainable: `.activate()`, `.pause(reason='manual')`, `.resume()`, `.archive()`.

In [ ]:
# profiles[0].pause(reason='manual')
# profiles[0].resume()
# profiles[0].archive()

## 3. Marketplace

Buy vetted alphas with wallet credits (1 credit = 1 VND). Top-up flow returns a VietQR URL — user scans with any Vietnam banking app, transfer is auto-detected by sepay (or manually approved by admin in Phase 1).

In [ ]:
# Browse top listings
page = tlx.market.search(min_sharpe=2.0, sort='sharpe', limit=10)
df_market = pd.DataFrame([{
    'slug': l.slug,
    'title': l.title,
    'sharpe': l.snapshot.sharpe,
    'drawdown': l.snapshot.drawdown,
    'lifetime_vnd': l.price_vnd('lifetime'),
    'sold': l.stats.purchase_count,
    'rating': l.stats.rating_avg,
} for l in page])
df_market

In [ ]:
# Wallet balance + recent transactions
wallet = tlx.wallet.balance()
print(f'Balance:  {wallet.credits_balance:,} VND')
print(f'Topped:   {wallet.lifetime_topup_credits:,} VND')
print(f'Spent:    {wallet.lifetime_spent_credits:,} VND')

txs = tlx.wallet.transactions(limit=10)
pd.DataFrame([t.model_dump() for t in txs])[['created_at', 'kind', 'amount', 'balance_after', 'note']]

### Top up via VietQR

The SDK returns a QR url — embed it in your app or open in a browser. Memo includes the user's Discord ID + topup PK for admin matching.

In [ ]:
from IPython.display import Image, display

topup = tlx.wallet.topup(amount_vnd=500_000)
print(f'Topup #{topup.topup.id}  memo={topup.memo}')
print(f'Bank: {topup.bank}')
display(Image(url=topup.qr_url))

### Buy a listing

After top-up succeeds, your wallet has credits. Buying transfers them to the seller minus 10% platform fee.

In [ ]:
# Uncomment when wallet has enough credits
# purchase = tlx.market.buy(slug=df_market.iloc[0]['slug'], license_type='lifetime')
# print(f'Bought purchase #{purchase.id}  charged={purchase.credits_charged:,} VND')
# print(f'Net to seller: {purchase.seller_credits:,} VND  fee: {purchase.platform_fee_credits:,} VND')

In [ ]:
# My licenses
licenses = tlx.market.library()
pd.DataFrame([l.model_dump() for l in licenses])[['alpha_id', 'source', 'license_type', 'expires_at', 'is_active']].head(20)

### List your own alpha for sale

Requires the alpha to pass all 5 overfit checks. Server re-validates server-side.

In [ ]:
# listing = tlx.market.list_for_sale(
#     alpha_id=result.alpha_id,
#     title=f'BTC mean reversion v1 (Sharpe {result.sharpe:.1f})',
#     description_md='Notes about the strategy in markdown.',
#     tags='btc, futures, mean-reversion',
#     lifetime_price_vnd=10_000_000,
#     lifetime_price_usd=400,
# )
# print(f'Listed at /market/alpha/{listing.slug}/')

### Seller dashboard

In [ ]:
stats = tlx.market.seller_stats()
print(f'Total revenue:  {stats["total_revenue_credits"]:,} VND')
print(f'Lifetime sales: {stats["lifetime_sales"]}')
print(f'Pending payout: {stats["pending_payout_credits"]:,} VND')
print(f'Wallet:         {stats["wallet"].credits_balance:,} VND')
print(f'Active listings: {len(stats["listings"])}')

## End-to-end research → deploy → monetize

The 3 surfaces compose: research an alpha, deploy it via trading desk for live PnL, then list it on marketplace once you have track record. Below is the full one-cell script:

In [ ]:
# Research
result = (
    Alpha('rank(close - ts_mean(close, 20)) * volume')
    .region('crypto_trade').universe('TOP19').decay(4)
    .simulate(tlx)
)
print(f'1. Researched alpha {result.alpha_id} Sharpe={result.sharpe:.2f}')

# Deploy (requires existing credential + Pro+ tier for live)
# profile = tlx.trading.profiles.create(
#     name='auto_deployed_v1', alpha_id=result.alpha_id,
#     exchange='binance', credential_id=42, mode='simulation', book_usd=200,
# ).activate()
# print(f'2. Deployed profile #{profile.id}')

# Monetize after track record
# if result.passes_overfit():
#     listing = tlx.market.list_for_sale(
#         alpha_id=result.alpha_id,
#         title=f'My alpha — Sharpe {result.sharpe:.1f}',
#         lifetime_price_vnd=5_000_000,
#     )
#     print(f'3. Listed at /market/alpha/{listing.slug}/')